In [1]:
import sys
import os

parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

from src.data.temporal_loader import load_temporal_splits_from_json

/usr/local/lib/python3.12/dist-packages/modelopt/torch/utils/import_utils.py:31: UserWarning: Failed to import apex plugin due to: AttributeError("module 'transformers.modeling_utils' has no attribute 'Conv1D'"). You may ignore this warning if you do not need this plugin.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/modelopt/torch/utils/import_utils.py:31: UserWarning: Failed to import huggingface plugin due to: AttributeError("module 'transformers.modeling_utils' has no attribute 'Conv1D'"). You may ignore this warning if you do not need this plugin.
  warnings.warn(
TensorRT-LLM is not installed. Please install TensorRT-LLM or set TRTLLM_PLUGINS_PATH to the directory containing libnvinfer_plugin_tensorrt_llm.so to use converters for torch.distributed ops


[11/02/2025-14:09:30] [TRT] [W] Functionality provided through tensorrt.plugin module is experimental.


In [2]:
import torch
from monai.transforms import (
    EnsureChannelFirstd, Orientationd, Spacingd,
    ToTensord, Resized, ResizeWithPadOrCropd, MapTransform
)
from monai.transforms import ResampleToMatchd
from monai.metrics import DiceMetric

In [3]:
# Assemble per-patient sequences
root = "/scratch/ejf9db/new T1_split"
splits = load_temporal_splits_from_json(root)
X_train, X_val, X_test = splits["train"], splits["val"], splits["test"]

In [17]:
x = X_train[2]

sample = {
    "img_t": x["images"][0],
    "mask_t": x["labels"][0],
    "img_tp1": x["images"][1],
    "mask_tp1": x["labels"][1]
}

In [7]:
common_pre = [
    Orientationd(keys=["img_t", "mask_t", "img_tp1", "mask_tp1"], axcodes="RAS"),
    # put them on a common spacing (use your real spacing)
    Spacingd(keys=["img_t", "mask_t", "img_tp1", "mask_tp1"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest", "bilinear", "nearest")),
    ToTensord(keys=["img_t", "mask_t", "img_tp1", "mask_tp1"]),
]

In [18]:
data = sample
for t in common_pre:
    data = t(data)

# now resample mask_t INTO img_tp1 space
resample_mask = ResampleToMatchd(
    keys="mask_t",
    key_dst="img_tp1",  # match this
    mode="nearest"
)

data = resample_mask(data)

pred_like_baseline = data["mask_t"]   # "prediction" = time0 mask in time1 space
target = data["mask_tp1"]

dice = DiceMetric(include_background=False, reduction="mean")
d = dice(pred_like_baseline.unsqueeze(0), target.unsqueeze(0))
print("no-reg ceiling dice:", float(d))

no-reg ceiling dice: 0.0


In [13]:
x = X_val[0]

sample = {
    "img_t": x["images"][0],
    "mask_t": x["labels"][0],
    "img_tp1": x["images"][1],
    "mask_tp1": x["labels"][1]
}

In [14]:
dice.reset()

data = sample
for t in common_pre:
    data = t(data)

# now resample mask_t INTO img_tp1 space
resample_mask = ResampleToMatchd(
    keys="mask_t",
    key_dst="img_tp1",  # match this
    mode="nearest"
)

data = resample_mask(data)

pred_like_baseline = data["mask_t"]   # "prediction" = time0 mask in time1 space
target = data["mask_tp1"]

dice = DiceMetric(include_background=False, reduction="mean")
d = dice(pred_like_baseline.unsqueeze(0), target.unsqueeze(0))
print("no-reg ceiling dice:", float(d))

no-reg ceiling dice: 0.0


In [24]:
import numpy as np
from tqdm import tqdm

dice_scores = []
for x in tqdm(X_train):
    sample = {
        "img_t": x["images"][0],
        "mask_t": x["labels"][0],
        "img_tp1": x["images"][1],
        "mask_tp1": x["labels"][1]
    }
    data = sample
    for t in common_pre:
        data = t(data)
    
    # now resample mask_t INTO img_tp1 space
    resample_mask = ResampleToMatchd(
        keys="mask_t",
        key_dst="img_tp1",  # match this
        mode="nearest"
    )
    
    data = resample_mask(data)
    
    pred_like_baseline = data["mask_t"]   # "prediction" = time0 mask in time1 space
    target = data["mask_tp1"]
    
    dice = DiceMetric(include_background=False, reduction="mean")
    d = dice(pred_like_baseline.unsqueeze(0), target.unsqueeze(0))
    if not np.isnan(float(d)):
        dice_scores.append(float(d))

100%|██████████| 109/109 [01:20<00:00,  1.35it/s]


In [25]:
print(f"mean no-reg ceiling: {np.mean(dice_scores)} +/- {np.std(dice_scores)}")

mean no-reg ceiling: 0.1452807541936636 +/- 0.24640325308429076
